## Feature Verification Analysis for Two Towers Model:

**Auth:** Uses `secrets/recosys-service-account.json` under the repo root (or `GOOGLE_APPLICATION_CREDENTIALS` if set). Run with the kernel **working directory** set to the repo root or to `notebooks/` so the key path resolves.

In [11]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.cloud import bigquery, storage
from google.oauth2 import service_account

# ── Config (aligned with scripts/build_*_features.py) ─────────────────────────
PROJECT_ID = "recosys-489001"
DATASET_ID = "recosys"
BUCKET_NAME = "recosys-data-bucket"

# Repo root: cwd is usually RecoSys or RecoSys/notebooks when running in VS Code / Jupyter
_cwd = Path.cwd().resolve()
_REPO_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
_DEFAULT_KEY = _REPO_ROOT / "secrets" / "recosys-service-account.json"
KEY_PATH = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS", str(_DEFAULT_KEY))

_credentials = service_account.Credentials.from_service_account_file(
    KEY_PATH,
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)
client = bigquery.Client(project=PROJECT_ID, credentials=_credentials)
gcs_client = storage.Client(project=PROJECT_ID, credentials=_credentials)

print(f"[ok] BigQuery — {_credentials.service_account_email}")
print(f"     project={client.project}  dataset={DATASET_ID}")

# Verify GCS (same credentials as BQ)
_bucket = gcs_client.bucket(BUCKET_NAME)
_bucket.reload()
print(f"[ok] GCS — gs://{BUCKET_NAME}/ (readable)")
_feature_blobs = list(gcs_client.list_blobs(BUCKET_NAME, prefix="features/", max_results=5))
if _feature_blobs:
    print("     sample under features/:")
    for _b in _feature_blobs:
        print(f"       {_b.name}")
else:
    print("     (no blobs under features/ in first 5 results — check export paths)")

# Pull item-level stats from interactions + item features
item_stats_sql = """
SELECT
  i.product_id,
  i.confidence_score                         AS total_confidence,
  i.n_views,
  i.n_purchases,
  i.n_carts,
  SAFE_DIVIDE(i.n_purchases, i.n_views)      AS purchase_rate,
  f.cat_l1,
  f.brand,
  f.price_bucket,
  f.avg_price,
  -- how many distinct users interacted with this item
  COUNT(DISTINCT i.user_id)                  AS n_users
FROM `recosys-489001.recosys.interactions_train_50k` i
JOIN `recosys-489001.recosys.item_features`          f USING (product_id)
GROUP BY 1,2,3,4,5,6,7,8,9,10
"""
items = client.query(item_stats_sql).to_dataframe()
print(f"Items with any interaction: {len(items):,}")
print(f"Items with at least 1 purchase: {(items.n_purchases > 0).sum():,}")
print(f"Items with zero purchases:      {(items.n_purchases == 0).sum():,}")

[ok] BigQuery — 921967012784-compute@developer.gserviceaccount.com
     project=recosys-489001  dataset=recosys
[ok] GCS — gs://recosys-data-bucket/ (readable)
     sample under features/:
       features/
       features/interactions_train_full/000000000000.parquet
       features/interactions_train_full/000000000001.parquet
       features/interactions_train_full/000000000002.parquet
       features/interactions_train_full/000000000003.parquet


c:\Users\Patron\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Items with any interaction: 211,597
Items with at least 1 purchase: 14,218
Items with zero purchases:      197,379


In [12]:
# Bin items by popularity (n_users) and see if purchase rate varies across bins.
# Raw n_users has massive ties (e.g. many items with 1 user), so pd.qcut(n_users)
# can fail with duplicate bin edges; ranking breaks ties so quintiles are well-defined.
items['popularity_bin'] = pd.qcut(
    items['n_users'].rank(method='first'),
    q=5,
    labels=['very_cold', 'cold', 'medium', 'warm', 'hot'],
)

pop_vs_purchase_n_users = items.groupby('popularity_bin', observed=True).agg(
    n_items        = ('product_id', 'count'),
    avg_n_users    = ('n_users', 'mean'),
    pct_purchased  = ('n_purchases', lambda x: (x > 0).mean()),
    avg_purchase_rate = ('purchase_rate', 'mean'),
).round(4)

# Same quintile logic for total_confidence (distinct signal from raw n_users).
items['confidence_bin'] = pd.qcut(
    items['total_confidence'].rank(method='first'),
    q=5,
    labels=['very_cold', 'cold', 'medium', 'warm', 'hot'],
)
pop_vs_purchase_confidence = items.groupby('confidence_bin', observed=True).agg(
    n_items        = ('product_id', 'count'),
    avg_confidence = ('total_confidence', 'mean'),
    pct_purchased  = ('n_purchases', lambda x: (x > 0).mean()),
    avg_purchase_rate = ('purchase_rate', 'mean'),
).round(4)

print("By n_users (distinct users interacting):")
print(pop_vs_purchase_n_users.to_string())
print()
print("By total_confidence:")
print(pop_vs_purchase_confidence.to_string())

By n_users (distinct users interacting):
                n_items  avg_n_users  pct_purchased  avg_purchase_rate
popularity_bin                                                        
very_cold         42320          1.0         0.0009             0.0000
cold              42319          1.0         0.0665             0.0473
medium            42319          1.0         0.2146             0.0512
warm              42319       2.1492         0.0408             0.0109
hot               42320      12.5922         0.0131             0.0064

By total_confidence:
                n_items  avg_confidence  pct_purchased  avg_purchase_rate
confidence_bin                                                           
very_cold         42320             1.0         0.0000             0.0000
cold              42319             1.0         0.0000             0.0000
medium            42319            1.86         0.0000             0.0000
warm              42319          3.2543         0.0071             0.0

In [13]:
# What fraction of all purchases come from the top N% of items by purchase_rate?
items_with_purchases = items[items.n_purchases > 0].copy()
items_with_purchases = items_with_purchases.sort_values('n_purchases', ascending=False)
items_with_purchases['cumulative_purchase_share'] = (
    items_with_purchases.n_purchases.cumsum() / items_with_purchases.n_purchases.sum()
)

# What % of items account for 50%, 80%, 90% of purchases?
thresholds = [0.5, 0.8, 0.9]
for t in thresholds:
    idx = (items_with_purchases.cumulative_purchase_share >= t).idxmax()
    rank = items_with_purchases.index.get_loc(idx) + 1
    pct  = rank / len(items_with_purchases) * 100
    print(f"Top {pct:.1f}% of items account for {t*100:.0f}% of purchases")

Top 26.2% of items account for 50% of purchases
Top 70.5% of items account for 80% of purchases
Top 85.2% of items account for 90% of purchases


In [14]:
# For each user, compute what % of their events fall in their top category
user_cat_sql = """
SELECT
  t.user_id,
  f.cat_l1,
  COUNT(*) AS n_events
FROM `recosys-489001.recosys.train_50k`        t
JOIN `recosys-489001.recosys.item_features`    f USING (product_id)
GROUP BY t.user_id, f.cat_l1
"""
user_cats = client.query(user_cat_sql).to_dataframe()

# Compute top-category concentration per user
total_per_user = user_cats.groupby('user_id')['n_events'].sum().rename('total')
user_cats = user_cats.join(total_per_user, on='user_id')
user_cats['share'] = user_cats['n_events'] / user_cats['total']

top_cat = user_cats.sort_values('share', ascending=False)\
                   .groupby('user_id').first()\
                   .reset_index()[['user_id','cat_l1','share']]

print("Distribution of top-category concentration per user:")
print(top_cat['share'].describe().round(3))
print()
print(pd.cut(top_cat['share'],
             bins=[0,.5,.7,.85,1.0],
             labels=['<50%','50-70%','70-85%','>85%'])
      .value_counts().sort_index())

c:\Users\Patron\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Distribution of top-category concentration per user:
count    44559.0
mean        0.79
std        0.215
min          0.2
25%        0.602
50%        0.833
75%          1.0
max          1.0
Name: share, dtype: Float64

share
<50%       6713
50-70%     9444
70-85%     6576
>85%      21826
Name: count, dtype: int64


In [15]:
# The definitive test: do users predominantly purchase in their top train category?
test_purchases_sql = """
SELECT
  t.user_id,
  f.cat_l1  AS purchased_cat
FROM `recosys-489001.recosys.test_50k`      t
JOIN `recosys-489001.recosys.item_features` f USING (product_id)
WHERE t.event_type = 'purchase'
"""
test_purchases = client.query(test_purchases_sql).to_dataframe()

# Join with user's top category from training
merged = test_purchases.merge(
    top_cat[['user_id','cat_l1']].rename(columns={'cat_l1':'train_top_cat'}),
    on='user_id',
    how='inner'
)

match_rate = (merged.purchased_cat == merged.train_top_cat).mean()
print(f"Rate at which Feb purchases match user's train top_category: {match_rate:.1%}")
print(f"  (If this were random across 13 categories, baseline would be ~7.7%)")
print()
print("Purchase category distribution vs. top-train-category match:")
print(merged.groupby('train_top_cat')['purchased_cat']
      .apply(lambda x: (x == x.name).mean())
      .round(3)
      .sort_values(ascending=False))

c:\Users\Patron\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Rate at which Feb purchases match user's train top_category: 49.1%
  (If this were random across 13 categories, baseline would be ~7.7%)

Purchase category distribution vs. top-train-category match:
train_top_cat
medicine        1.000
electronics     0.676
sport           0.359
unknown         0.318
kids            0.303
computers       0.297
furniture       0.261
construction    0.250
appliances      0.171
apparel         0.077
auto            0.062
accessories     0.000
country_yard    0.000
stationery      0.000
Name: purchased_cat, dtype: float64


In [16]:
# Check if users have consistent purchase price behavior
price_sql = """
SELECT
  t.user_id,
  t.event_type,
  f.price_bucket,
  f.avg_price
FROM `recosys-489001.recosys.train_50k`      t
JOIN `recosys-489001.recosys.item_features`  f USING (product_id)
WHERE t.event_type = 'purchase'
"""
price_data = client.query(price_sql).to_dataframe()

# Only users with 2+ purchases have meaningful price consistency
purchase_counts = price_data.groupby('user_id').size()
multi_buyers = purchase_counts[purchase_counts >= 2].index
price_data_multi = price_data[price_data.user_id.isin(multi_buyers)]

# Within each user, compute std deviation of price_bucket across their purchases
user_price_std = price_data_multi.groupby('user_id')['price_bucket'].std()

print(f"Users with 2+ purchases: {len(multi_buyers):,}")
print(f"\nPer-user std dev of price_bucket (0=always same tier, 3.5=totally random):")
print(user_price_std.describe().round(3))
print()
print("Distribution of price consistency:")
print(pd.cut(user_price_std,
             bins=[-0.01, 0.5, 1.0, 2.0, 4.0],
             labels=['very_consistent','consistent','mixed','random'])
      .value_counts().sort_index())

c:\Users\Patron\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Users with 2+ purchases: 3,921

Per-user std dev of price_bucket (0=always same tier, 3.5=totally random):
count    3921.0
mean      0.704
std        0.68
min         0.0
25%         0.0
50%       0.603
75%       1.155
max       4.243
Name: price_bucket, dtype: Float64

Distribution of price consistency:
price_bucket
very_consistent    1579
consistent         1217
mixed               902
random              222
Name: count, dtype: int64


In [17]:
# Does a user's train avg_purchase_price predict their test purchase price tier?
train_price_sql = """
SELECT
  t.user_id,
  AVG(f.avg_price) AS train_avg_purchase_price
FROM `recosys-489001.recosys.train_50k`      t
JOIN `recosys-489001.recosys.item_features`  f USING (product_id)
WHERE t.event_type = 'purchase'
GROUP BY t.user_id
"""
test_price_sql = """
SELECT
  t.user_id,
  AVG(f.avg_price) AS test_avg_purchase_price
FROM `recosys-489001.recosys.test_50k`       t
JOIN `recosys-489001.recosys.item_features`  f USING (product_id)
WHERE t.event_type = 'purchase'
GROUP BY t.user_id
"""

train_price = client.query(train_price_sql).to_dataframe()
test_price  = client.query(test_price_sql).to_dataframe()

both = train_price.merge(test_price, on='user_id')
corr = both['train_avg_purchase_price'].corr(both['test_avg_purchase_price'])
print(f"Pearson correlation between train and test avg purchase price: {corr:.3f}")
print(f"  (>0.5 = strong stability, worth including as feature)")
print(f"  (0.2–0.5 = moderate, worth including)")
print(f"  (<0.2 = unstable, likely noise)")
print(f"\nSample size: {len(both):,} users with purchases in both train and test")

c:\Users\Patron\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\cloud\bigquery\table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Pearson correlation between train and test avg purchase price: 0.462
  (>0.5 = strong stability, worth including as feature)
  (0.2–0.5 = moderate, worth including)
  (<0.2 = unstable, likely noise)

Sample size: 1,084 users with purchases in both train and test


In [18]:
# Compile results into a decision table (run cells 1–7 first)
_BIN_ORDER = ['very_cold', 'cold', 'medium', 'warm', 'hot']


def _pct_purchased_monotonic(pop_df: pd.DataFrame) -> bool:
    s = pop_df.reindex(_BIN_ORDER)['pct_purchased']
    return bool(s.is_monotonic_increasing)


def _fmt_mono(ok: bool) -> str:
    return 'monotonic ↑ pct_purchased' if ok else 'not monotonic'


def _yes_no(cond: bool) -> str:
    return 'yes' if cond else 'no'


mono_n_users = _pct_purchased_monotonic(pop_vs_purchase_n_users)
mono_conf = _pct_purchased_monotonic(pop_vs_purchase_confidence)

pct_zero_purchase_items = (items['n_purchases'] == 0).mean() * 100
include_purchase_rate_feat = pct_zero_purchase_items > 40

include_top_cat = match_rate > 0.30
include_avg_price = corr > 0.4

r_conf = _fmt_mono(mono_conf)
r_users = _fmt_mono(mono_n_users)
r_zero = f'{pct_zero_purchase_items:.1f}% items with 0 purchases'
r_cat = f'{match_rate:.1%} test purchases match train top category'
r_price = f'ρ = {corr:.3f} (train vs test avg purchase price)'

inc_conf = _yes_no(mono_conf)
inc_users = _yes_no(mono_n_users)
inc_pr = _yes_no(include_purchase_rate_feat)
inc_cat = _yes_no(include_top_cat)
inc_price = _yes_no(include_avg_price)

print('\n' + '=' * 60)
print('FEATURE VERIFICATION SUMMARY')
print('=' * 60)
print(f"""
Feature                    | Test Performed              | Result                                      | Include?
---------------------------|-----------------------------|---------------------------------------------|---------
item_total_confidence      | quintiles by confidence     | {r_conf:<43} | {inc_conf:<8}
item_n_users               | quintiles by n_users        | {r_users:<43} | {inc_users:<8}
item_purchase_rate         | % items with 0 purchases    | {r_zero:<43} | {inc_pr:<8}
user top_category          | category match rate in test | {r_cat:<43} | {inc_cat:<8}
user avg_purchase_price    | train/test price correlation| {r_price:<43} | {inc_price:<8}

Decision threshold (from notebook):
  - item popularity: include if pct_purchased rises monotonically across bins
  - item_purchase_rate: include if >40% of items have 0 purchases
  - top_category: include if match_rate > 30%
  - avg_purchase_price: include if train/test correlation > 0.4
""")


FEATURE VERIFICATION SUMMARY

Feature                    | Test Performed              | Result                                      | Include?
---------------------------|-----------------------------|---------------------------------------------|---------
item_total_confidence      | quintiles by confidence     | monotonic ↑ pct_purchased                   | yes     
item_n_users               | quintiles by n_users        | not monotonic                               | no      
item_purchase_rate         | % items with 0 purchases    | 93.3% items with 0 purchases                | yes     
user top_category          | category match rate in test | 49.1% test purchases match train top category | yes     
user avg_purchase_price    | train/test price correlation| ρ = 0.462 (train vs test avg purchase price) | yes     

Decision threshold (from notebook):
  - item popularity: include if pct_purchased rises monotonically across bins
  - item_purchase_rate: include if >40% of items have